# AI-Powered Demand Forecasting & Inventory Optimization

## Exploratory Data Analysis

**Notebook:** `01_exploratory_data_analysis.ipynb`

**Dataset:** M5 Forecasting - Accuracy (Walmart retail data)

**Analysis Date:** 2026-08-31


## 1. Business Objective

### What the M5 Dataset Represents
The M5 dataset is a publicly available retail sales forecasting dataset published by Walmart for the Kaggle M5 Forecasting competition. It contains daily sales data for 30,490 products across 10 stores in three US states (California, Texas, and Wisconsin), spanning approximately 5.5 years (January 2011 to April 2016).

### What Demand Forecasting Means
Demand forecasting is the process of predicting future customer demand for products or services. In retail and supply-chain contexts, accurate demand forecasts enable businesses to:
- Maintain optimal inventory levels
- Reduce stockouts and overstock situations
- Improve customer satisfaction
- Minimize holding costs and waste
- Optimize supply-chain operations

### Why Demand Patterns Matter for Inventory Planning
Understanding demand patterns is essential for effective inventory planning. Products with stable, predictable demand require different inventory strategies than products with intermittent or highly variable demand.

### Project Objective
This project aims to build an AI-powered demand forecasting and inventory optimization system. The immediate objective of this notebook is to explore the M5 dataset, understand demand patterns, identify key characteristics of the data, and generate actionable business insights.


## Setup

Import libraries and configure the environment.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
RANDOM_STATE = 42

# Configure plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Determine project root (two levels up from this notebook)
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'demand_dev.parquet'

print('Project root:', PROJECT_ROOT)
print('Data path:', DATA_PATH)
print('Data exists:', DATA_PATH.exists())


## 6. Product Analysis

Analyze demand patterns at the product level.


In [ ]:
# Calculate product-level metrics
product_stats = df.groupby('id')['demand'].agg(['sum', 'mean', 'std']).reset_index()
product_stats.columns = ['id', 'total_demand', 'avg_demand', 'std_demand']

# Calculate coefficient of variation (handle zero mean)
product_stats['cv'] = np.where(
    product_stats['avg_demand'] > 0,
    product_stats['std_demand'] / product_stats['avg_demand'],
    np.nan
)

# Calculate zero-demand percentage
zero_pct = df.groupby('id')['demand'].apply(lambda x: (x == 0).mean() * 100).reset_index()
zero_pct.columns = ['id', 'zero_pct']

# Merge statistics
product_stats = product_stats.merge(zero_pct, on='id')
product_stats = product_stats.sort_values('total_demand', ascending=False)

print('Product Statistics Summary:')
print('=' * 60)
print(f'Total products: {len(product_stats)}')
print(f'\nTop 10 Products by Total Demand:')
print(product_stats.head(10).to_string(index=False))
print(f'\nBottom 10 Products by Total Demand:')
print(product_stats.tail(10).to_string(index=False))


In [ ]:
# Create charts for top and bottom products
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 products
top_10 = product_stats.head(10)
axes[0].barh(range(len(top_10)), top_10['total_demand'], edgecolor='black')
axes[0].set_yticks(range(len(top_10)))
axes[0].set_yticklabels(top_10['id'], fontsize=9)
axes[0].set_title('Top 10 Products by Total Demand', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Demand (units)')
axes[0].invert_yaxis()

# Bottom 10 products
bottom_10 = product_stats.tail(10)
axes[1].barh(range(len(bottom_10)), bottom_10['total_demand'], edgecolor='black')
axes[1].set_yticks(range(len(bottom_10)))
axes[1].set_yticklabels(bottom_10['id'], fontsize=9)
axes[1].set_title('Bottom 10 Products by Total Demand', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Demand (units)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 7. Store Analysis

Analyze demand patterns by store.


In [ ]:
# Calculate store-level metrics
store_stats = df.groupby('store_id')['demand'].agg(['sum', 'mean']).reset_index()
store_stats.columns = ['store_id', 'total_demand', 'avg_daily_demand']
store_stats = store_stats.sort_values('total_demand', ascending=False)

# Create bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Total demand by store
axes[0].bar(store_stats['store_id'], store_stats['total_demand'], edgecolor='black')
axes[0].set_title('Total Demand by Store', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Store')
axes[0].set_ylabel('Total Demand (units)')
axes[0].tick_params(axis='x', rotation=45)

# Average daily demand by store
axes[1].bar(store_stats['store_id'], store_stats['avg_daily_demand'], edgecolor='black')
axes[1].set_title('Average Daily Demand by Store', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Store')
axes[1].set_ylabel('Average Daily Demand (units)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nStore Statistics:')
print(store_stats.to_string(index=False))


## 8. Category & Department Analysis

Analyze demand by product category and department.


In [ ]:
# Category analysis
cat_stats = df.groupby('cat_id')['demand'].agg(['sum', 'mean']).reset_index()
cat_stats.columns = ['category', 'total_demand', 'avg_demand']
cat_stats = cat_stats.sort_values('total_demand', ascending=False)

# Department analysis
dept_stats = df.groupby('dept_id')['demand'].agg(['sum', 'mean']).reset_index()
dept_stats.columns = ['department', 'total_demand', 'avg_demand']
dept_stats = dept_stats.sort_values('total_demand', ascending=False)

# Create charts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Category chart
axes[0].bar(cat_stats['category'], cat_stats['total_demand'], edgecolor='black')
axes[0].set_title('Total Demand by Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Total Demand (units)')

# Department chart
axes[1].bar(dept_stats['department'], dept_stats['total_demand'], edgecolor='black')
axes[1].set_title('Total Demand by Department', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Department')
axes[1].set_ylabel('Total Demand (units)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nCategory Statistics:')
print(cat_stats.to_string(index=False))
print('\nDepartment Statistics:')
print(dept_stats.to_string(index=False))


## 9. Demand Volatility

Analyze demand variability and identify products with different demand patterns.


In [ ]:
# Classify products based on coefficient of variation and zero-demand percentage
cv_median = product_stats['cv'].median()
zero_median = product_stats['zero_pct'].median()

print(f'CV Median: {cv_median:.2f}')
print(f'Zero-Demand % Median: {zero_median:.2f}%')
print()

# Identify product types
stable = product_stats[(product_stats['cv'] <= cv_median) & (product_stats['zero_pct'] <= zero_median)]
volatile = product_stats[(product_stats['cv'] > cv_median) & (product_stats['zero_pct'] <= zero_median)]
intermittent = product_stats[product_stats['zero_pct'] > zero_median]

print(f'Stable demand products: {len(stable)} ({len(stable)/len(product_stats)*100:.1f}%)')
print(f'Volatile demand products: {len(volatile)} ({len(volatile)/len(product_stats)*100:.1f}%)')
print(f'Intermittent demand products: {len(intermittent)} ({len(intermittent)/len(product_stats)*100:.1f}%)')


In [ ]:
# Show examples of each type
print('Examples of Stable Demand Products:')
print(stable.head(5)[['id', 'total_demand', 'avg_demand', 'cv', 'zero_pct']].to_string(index=False))

print('\nExamples of Volatile Demand Products:')
print(volatile.head(5)[['id', 'total_demand', 'avg_demand', 'cv', 'zero_pct']].to_string(index=False))

print('\nExamples of Intermittent Demand Products:')
print(intermittent.head(5)[['id', 'total_demand', 'avg_demand', 'cv', 'zero_pct']].to_string(index=False))


## 4. Demand Over Time

Analyze how demand changes over the analysis period.


In [ ]:
# Aggregate demand by date
daily_demand = df.groupby('date')['demand'].sum().reset_index()

# Create time-series plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_demand['date'], daily_demand['demand'], linewidth=0.8, alpha=0.8)
ax.set_title('Total Daily Demand Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Demand (units)', fontsize=12)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('Daily demand statistics:')
print(f'  Mean: {daily_demand[\"demand\"].mean():,.0f} units')
print(f'  Std:  {daily_demand[\"demand\"].std():,.0f} units')


In [ ]:
# Aggregate demand by week
df['year_week'] = df['date'].dt.to_period('W').dt.start_time
weekly_demand = df.groupby('year_week')['demand'].sum().reset_index()

# Create weekly time-series plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(weekly_demand['year_week'], weekly_demand['demand'], linewidth=1.2)
ax.set_title('Total Weekly Demand Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Total Demand (units)', fontsize=12)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


### Observations on Demand Over Time

- **Trend**: The data shows the overall demand trend across the analysis period.
- **Seasonality**: Regular patterns may be visible at annual or weekly frequencies.
- **Spikes**: Unusual demand spikes may correspond to holidays, promotions, or events.
- **Structural changes**: Shifts in demand levels may indicate changes in business operations or product mix.


## 5. Weekly Seasonality

Analyze how demand varies by day of the week.


In [ ]:
# Aggregate demand by weekday
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_demand = df.groupby('weekday')['demand'].mean().reindex(weekday_order).reset_index()
weekday_demand.columns = ['weekday', 'avg_demand']

# Create bar chart
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(weekday_demand['weekday'], weekday_demand['avg_demand'], edgecolor='black')
ax.set_title('Average Demand by Weekday', fontsize=14, fontweight='bold')
ax.set_xlabel('Weekday', fontsize=12)
ax.set_ylabel('Average Demand (units)', fontsize=12)
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print('\nAverage Demand by Weekday:')
print(weekday_demand.to_string(index=False))


### Weekday Effect Interpretation

The bar chart shows how average demand varies by day of the week. Differences in weekday demand may reflect:

- **Shopping patterns**: Customers may shop more on certain days (e.g., weekends vs. weekdays).
- **Promotional activities**: Stores may run promotions on specific days.
- **Pay cycles**: Consumer purchasing may align with weekly or bi-weekly pay schedules.

The magnitude of weekday effects should be evaluated relative to overall demand variability to determine their practical significance.


## 10. Price Analysis

Analyze the distribution and patterns of selling prices.


In [ ]:
# Price statistics
price_stats = df['sell_price'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
missing_pct = df['sell_price'].isna().mean() * 100

print('Price Statistics:')
print('=' * 40)
print(price_stats)
print(f'\nMissing prices: {missing_pct:.2f}%')


In [ ]:
# Create price distribution plot
fig, ax = plt.subplots(figsize=(12, 6))
prices = df['sell_price'].dropna()
ax.hist(prices, bins=50, edgecolor='black', alpha=0.7)
ax.set_title('Distribution of Selling Prices', fontsize=14, fontweight='bold')
ax.set_xlabel('Price ($)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.axvline(prices.mean(), color='red', linestyle='--', label=f'Mean: ${prices.mean():.2f}')
ax.axvline(prices.median(), color='green', linestyle='--', label=f'Median: ${prices.median():.2f}')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Analyze price changes for a few representative products
sample_products = df['id'].unique()[:3]

fig, ax = plt.subplots(figsize=(14, 6))

for product in sample_products:
    product_data = df[df['id'] == product].sort_values('date')
    ax.plot(product_data['date'], product_data['sell_price'], label=product, alpha=0.7)

ax.set_title('Price Trends for Sample Products', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price ($)', fontsize=12)
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 11. Price vs Demand

Investigate the relationship between price and demand.


In [ ]:
# Calculate correlation (excluding missing prices)
price_demand = df[['sell_price', 'demand']].dropna()
correlation = price_demand['sell_price'].corr(price_demand['demand'])

print(f'Price-Demand Correlation: {correlation:.4f}')
print(f'Number of observations: {len(price_demand):,}')


In [ ]:
# Create scatter plot with sampled data
sample_size = min(10000, len(price_demand))
sample = price_demand.sample(sample_size, random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(sample['sell_price'], sample['demand'], alpha=0.3, s=10)
ax.set_title(f'Price vs Demand (n={sample_size:,})', fontsize=14, fontweight='bold')
ax.set_xlabel('Price ($)', fontsize=12)
ax.set_ylabel('Demand (units)', fontsize=12)

# Add trend line
z = np.polyfit(sample['sell_price'], sample['demand'], 1)
p = np.poly1d(z)
x_line = np.linspace(sample['sell_price'].min(), sample['sell_price'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.2f})')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nNote: Scatter plot uses a random sample of {sample_size:,} observations for readability.')


### Limitations of Price-Demand Correlation

The observed correlation between price and demand should be interpreted cautiously:

1. **Confounding factors**: Demand is influenced by many factors beyond price (seasonality, promotions, competition).
2. **Endogeneity**: Prices may be set in response to expected demand, creating reverse causality.
3. **Aggregation bias**: Daily aggregation may obscure within-day price-demand dynamics.
4. **Product heterogeneity**: Different products may have different price sensitivities.

Correlation does not imply causation. Any price-demand relationship should be validated with proper causal inference methods.
